In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 74.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=c0840a07e1c4fda9a4b2a4cfc3aef9bf95c25c7a0f68da6df69c37a00d89938f
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [3]:
def get_quantum_random_bits(n):
    """Generates n random bits by measuring a single qubit n times."""
    bits = []
    for _ in range(n):
        # Create a 1-qubit circuit to get 1 random bit
        qc = QuantumCircuit(1, 1)
        qc.h(0) # Put in superposition
        qc.measure(0, 0)

        backend = BasicSimulator()
        job = backend.run(transpile(qc, backend), shots=1)
        bitstring = list(job.result().get_counts().keys())[0]
        bits.append(int(bitstring))
    return bits

# Now you can safely use N_BITS = 40 or higher
N_BITS = 40

In [4]:
# --- ALICE ---
alice_bits = get_quantum_random_bits(N_BITS)
alice_bases = get_quantum_random_bits(N_BITS)

# --- EVE, BOB, AND MEASUREMENT ---
bob_results = []
bob_bases = get_quantum_random_bits(N_BITS)
eve_bases = get_quantum_random_bits(N_BITS)

for i in range(N_BITS):
    # Alice's State
    qc = QuantumCircuit(1, 1)
    if alice_bits[i] == 1: qc.x(0)
    if alice_bases[i] == 1: qc.h(0)

    # EVE INTERCEPTS [cite: 9]
    if eve_bases[i] == 1: qc.h(0)
    qc.measure(0, 0) # Eve collapses the state [cite: 10]

    # BOB RECEIVES (The state is already collapsed by Eve)
    if bob_bases[i] == 1: qc.h(0)
    qc.measure(0, 0)

    backend = BasicSimulator()
    res = int(list(backend.run(transpile(qc, backend), shots=1).result().get_counts().keys())[0])
    bob_results.append(res)

# --- ANALYSIS ---
sifted_indices = [i for i in range(N_BITS) if alice_bases[i] == bob_bases[i]]
errors = sum(1 for i in sifted_indices if alice_bits[i] != bob_results[i])
error_rate = errors / len(sifted_indices) if sifted_indices else 0

print(f"Error rate: {error_rate:.2%}")
if error_rate > 0.15: # Standard threshold [cite: 10]
    print("ALERT: Attacker detected!")

Error rate: 26.32%
ALERT: Attacker detected!
